<a href="https://colab.research.google.com/github/nitsundon/Load-Forecast/blob/main/LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install plotly tensorflow scikit-learn

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# LSTM Forecasting with Last Week Feature + Plotly Plotting

# Step 1: Import Libraries
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

# Step 2: Load DataFrame from Pickle
df = pd.read_pickle('/content/drive/MyDrive/Libraries/pickle/preprocessed_demand_df.pkl')

# Step 3: Ensure datetime index and sorted
df = df.sort_index()
if not isinstance(df.index, pd.DatetimeIndex):
    df.set_index('datetime', inplace=True)

# Step 4: Add 'same time last week' feature (672 steps = 15-min * 7 days)
df['last_week'] = df['demand'].shift(672)
df.dropna(inplace=True)

# Step 5: Scale features
scaler = MinMaxScaler()
df[['demand_scaled', 'last_week_scaled']] = scaler.fit_transform(df[['demand', 'last_week']])

# Step 6: Create sequences for LSTM
def create_sequences_with_last_week(data, seq_length):
    X, y = [], []
    timestamps = []
    for i in range(seq_length, len(data)):
        past_seq = data.iloc[i-seq_length:i]['demand_scaled'].values
        last_week_val = data.iloc[i]['last_week_scaled']
        sequence = np.column_stack((past_seq, np.full(seq_length, last_week_val)))
        X.append(sequence)
        y.append(data.iloc[i]['demand_scaled'])
        timestamps.append(data.index[i])
    return np.array(X), np.array(y), timestamps

seq_len = 24  # 6 hours of past data
X, y, timestamps = create_sequences_with_last_week(df, seq_len)

# Step 7: Train-test split
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]
test_timestamps = timestamps[split:]

# Step 8: Define LSTM Model
model = Sequential([
    LSTM(64, activation='relu', input_shape=(seq_len, 2)),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')

# Step 9: Train the model
model.fit(X_train, y_train, epochs=20, batch_size=32,
          validation_data=(X_test, y_test), verbose=1)

# Step 10: Predict and inverse transform
# Step 10: Predict and inverse transform
y_pred = model.predict(X_test)

# Reshape y_pred to have the same number of features as the scaler was fitted on
y_pred_reshaped = np.repeat(y_pred, 2, axis=1)  # Duplicate the predictions for both features
y_pred_rescaled = scaler.inverse_transform(y_pred_reshaped)[:, 0]  # Select the first column (demand) after inverse transform

# Reshape y_test to have 2 features before inverse transforming
y_test_reshaped = np.repeat(y_test.reshape(-1, 1), 2, axis=1)
y_test_rescaled = scaler.inverse_transform(y_test_reshaped)[:, 0]  # Select the first column (demand) after inverse transform




# Step 11: Plot using Plotly
fig = go.Figure()
fig.add_trace(go.Scatter(x=test_timestamps, y=y_test_rescaled.flatten(),
                         mode='lines', name='Actual Demand', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=test_timestamps, y=y_pred_rescaled.flatten(),
                         mode='lines', name='Predicted Demand', line=dict(color='red')))

fig.update_layout(title="LSTM Electricity Demand Forecast (with Last Week Feature)",
                  xaxis_title="Timestamp",
                  yaxis_title="Demand (MW)",
                  legend=dict(x=0, y=1),
                  height=500,
                  template="plotly_white")

fig.show()
